In [0]:
USE CATALOG `retail-dwh-project`;
USE SCHEMA gold;

CREATE OR REPLACE VIEW gold.monthly_sales_revenue AS
SELECT
    DATE_FORMAT(f.TxnDate, 'yyyy-MM') AS Month,
    COUNT(f.TransactionID) AS Total_Transactions,
    SUM(f.Quantity) AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue
FROM `retail-dwh-project`.silver.FactSales f
GROUP BY DATE_FORMAT(f.TxnDate, 'yyyy-MM')
ORDER BY Month;

SELECT * FROM gold.monthly_sales_revenue;

CREATE OR REPLACE VIEW gold.revenue_by_category AS
SELECT
    p.Category,
    COUNT(f.TransactionID) AS Total_Transactions,
    SUM(f.Quantity) AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue,
    ROUND(AVG(f.Amount), 2) AS Avg_Order_Value
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimProduct p 
    ON f.ProductSK = p.ProductSK
GROUP BY p.Category
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.revenue_by_category;


CREATE OR REPLACE VIEW gold.top10_products AS
SELECT
    p.ProductID,
    p.ProductName,
    p.Category,
    p.UnitPrice,
    SUM(f.Quantity) AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimProduct p 
    ON f.ProductSK = p.ProductSK
GROUP BY p.ProductID, p.ProductName, p.Category, p.UnitPrice
ORDER BY Total_Revenue DESC
LIMIT 10;

SELECT * FROM gold.top10_products;


CREATE OR REPLACE VIEW gold.store_sales_performance AS
SELECT
    st.StoreID,
    st.StoreName,
    st.Region,
    COUNT(f.TransactionID) AS Total_Transactions,
    SUM(f.Quantity) AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue,
    ROUND(AVG(f.Amount), 2) AS Avg_Transaction_Value
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimStore st 
    ON f.StoreSK = st.StoreSK
GROUP BY st.StoreID, st.StoreName, st.Region
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.store_sales_performance;



CREATE OR REPLACE VIEW gold.region_revenue_summary AS
SELECT
    st.Region,
    COUNT(DISTINCT st.StoreID) AS Total_Stores,
    COUNT(f.TransactionID) AS Total_Transactions,
    SUM(f.Quantity) AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimStore st 
    ON f.StoreSK = st.StoreSK
GROUP BY st.Region
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.region_revenue_summary;


CREATE OR REPLACE VIEW gold.top10_customers AS
SELECT
    c.CustomerID,
    c.CustomerName,
    c.City,
    COUNT(f.TransactionID) AS Total_Orders,
    SUM(f.Quantity) AS Total_Units_Bought,
    ROUND(SUM(f.Amount), 2) AS Total_Spent
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimCustomer c
    ON f.CustomerSK = c.CustomerSK
    AND c.IsActive = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
ORDER BY Total_Spent DESC
LIMIT 10;

SELECT * FROM gold.top10_customers;


CREATE OR REPLACE VIEW gold.monthly_revenue_by_region AS
SELECT
    DATE_FORMAT(f.TxnDate, 'yyyy-MM') AS Month,
    st.Region,
    COUNT(f.TransactionID) AS Total_Transactions,
    ROUND(SUM(f.Amount), 2) AS Total_Revenue
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimStore st 
    ON f.StoreSK = st.StoreSK
GROUP BY DATE_FORMAT(f.TxnDate, 'yyyy-MM'), st.Region
ORDER BY Month, Total_Revenue DESC;

SELECT * FROM gold.monthly_revenue_by_region;


CREATE OR REPLACE VIEW gold.customer_change_history AS
SELECT
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    CASE IsActive WHEN 1 THEN 'Active' ELSE 'Expired' END AS Status
FROM `retail-dwh-project`.silver.DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM `retail-dwh-project`.silver.DimCustomer
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;

SELECT * FROM gold.customer_change_history;



-- Q1: Highest revenue month
SELECT DATE_FORMAT(TxnDate,'yyyy-MM') AS Month,
       ROUND(SUM(Amount),2) AS Total_Revenue
FROM `retail-dwh-project`.silver.FactSales
GROUP BY Month
ORDER BY Total_Revenue DESC
LIMIT 1;

-- Q2: City with most active customers
SELECT City, COUNT(*) AS Customer_Count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE IsActive = 1
GROUP BY City
ORDER BY Customer_Count DESC
LIMIT 5;

-- Q3: Avg revenue per region
SELECT st.Region,
       ROUND(AVG(f.Amount),2) AS Avg_Revenue
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimStore st
ON f.StoreSK = st.StoreSK
GROUP BY st.Region;

-- Q4: Unsold products
SELECT p.*
FROM `retail-dwh-project`.silver.DimProduct p
LEFT JOIN `retail-dwh-project`.silver.FactSales f
ON p.ProductSK = f.ProductSK
WHERE f.ProductSK IS NULL;

-- Q5: Category revenue %
SELECT p.Category,
       ROUND(SUM(f.Amount),2) AS Revenue,
       ROUND(SUM(f.Amount)*100.0/
            SUM(SUM(f.Amount)) OVER(),2) AS Revenue_Pct
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimProduct p
ON f.ProductSK = p.ProductSK
GROUP BY p.Category;

-- Q6: Customers > 3 orders
SELECT c.CustomerID, c.CustomerName, c.City,
       COUNT(f.TransactionID) AS Orders
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimCustomer c
ON f.CustomerSK = c.CustomerSK AND c.IsActive = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
HAVING Orders > 3;

-- Q7: Weekly trend
SELECT WEEKOFYEAR(TxnDate) AS Week,
       COUNT(*) AS Transactions,
       ROUND(SUM(Amount),2) AS Revenue
FROM `retail-dwh-project`.silver.FactSales
GROUP BY Week
ORDER BY Week;




